##### About this notebook:

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------
# Author:             Erick Rico Esparza
# Dates:              Oct 29 - Nov 12, 2025
# Description:        Generate corrected composites for 500 & 850 hPa following paper-style (contours = H′, shading = H′)
#-----------------------------------------------------------------------------------------------------------------------------

# Week 6

## 1. Libraries and setup

In [12]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import ttest_ind
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from xarray.coding.variables import SerializationWarning
import warnings
warnings.filterwarnings("ignore", category=SerializationWarning)
import matplotlib
matplotlib.use("Agg")
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker

In [3]:
# --- Set display formatting
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# --- Domain & constants
LON_MIN, LON_MAX = -120, -85
LAT_MIN, LAT_MAX = 12, 33
LON_CDMX, LAT_CDMX = -99.13, 19.43

# MCMA box
SW_lat, SW_lon = 18.3, -100.9
NE_lat, NE_lon = 20.7, -97.4
MCMA_BOX = (SW_lon, SW_lat, NE_lon - SW_lon, NE_lat - SW_lat)

pollutants = ["PM2.5", "PM10", "O3", "NO2", "SO2"]

## 2. Load NARR data (500 and 850 hPa)

In [5]:
# 500 hPa
H500 = xr.open_dataset("hgt500_mex_2012_2024.nc")["hgt"]
U500 = xr.open_dataset("uwnd500_mex_2012_2024.nc")["uwnd"]
V500 = xr.open_dataset("vwnd500_mex_2012_2024.nc")["vwnd"]

# 850 hPa
H850 = xr.open_dataset("hgt850_mex_2012_2024.nc")["hgt"]
U850 = xr.open_dataset("uwnd850_mex_2012_2024.nc")["uwnd"]
V850 = xr.open_dataset("vwnd850_mex_2012_2024.nc")["vwnd"]

lon2d, lat2d = H500["lon"].values, H500["lat"].values

## 3. Loading pollutant data (CDMX CSV)

In [6]:
df = pd.read_csv("cdmx_citymean_daily_2012_2024.csv")
df["DATE"] = pd.to_datetime(df["DATE"])

# --- Defining monthly p90 events (consistent with Week 5) ---
def top10_by_month(series: pd.Series) -> pd.DatetimeIndex:
    ev = []
    for _, s in series.groupby(series.index.to_period("M")):
        if len(s) == 0:
            continue
        thr = s.quantile(0.90)
        ev += s[s >= thr].index.tolist()
    return pd.DatetimeIndex(sorted(set(ev)))

# Converting to time index for grouping
df = df.set_index("DATE").sort_index()

# Verifying data type
print(df.info())

events = {p: top10_by_month(df[p].dropna()) for p in pollutants}

print("\n>>> Event day count by pollutant (monthly p90):")
for p in pollutants:
    print(f"{p:5s}: {len(events[p])} days")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4555 entries, 2012-01-01 to 2024-12-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   NO2     4555 non-null   float64
 1   O3      4555 non-null   float64
 2   PM10    4555 non-null   float64
 3   PM2.5   4555 non-null   float64
 4   SO2     4555 non-null   float64
dtypes: float64(5)
memory usage: 213.5 KB
None

>>> Event day count by pollutant (monthly p90):
PM2.5: 549 days
PM10 : 542 days
O3   : 548 days
NO2  : 545 days
SO2  : 549 days


## 4. Daily climatology (smooth 31-day rolling)

In [ ]:
def daily_climatology(da):
    da = da.assign_coords(time=pd.to_datetime(da.time.values))
    da_noleap = da.sel(time=~((da.time.dt.month==2) & (da.time.dt.day==29)))
    clim = da_noleap.groupby("time.dayofyear").mean("time")
    clim = clim.rolling(dayofyear=31, center=True, min_periods=1).mean()
    return clim

clim500 = daily_climatology(H500)
clim850 = daily_climatology(H850) # for future use

In [8]:
# Helper functions for composites & plots
def composite(ds, dates):
    dates = pd.to_datetime(dates)
    valid = dates[(dates >= pd.to_datetime(ds.time.min().values)) & (dates <= pd.to_datetime(ds.time.max().values))]
    return ds.sel(time=valid).mean("time", skipna=True)

def ttest_mask(Hprime, dates):
    evt = Hprime.sel(time=Hprime.time.isin(dates))
    ctrl = Hprime.sel(time=~Hprime.time.isin(dates))
    t, p = ttest_ind(evt, ctrl, axis=0, equal_var=False, nan_policy="omit")
    return xr.DataArray(p < 0.05, coords=evt.isel(time=0).coords)

def plot_composite(lon, lat, Hm, Hp, U, V, sig, pollutant, level):
    """
    Stable Matplotlib-only version (no draw, no clabel, no PolyCollection bugs).
    Composite plot anomalies for a given pollutant and pressure level
    - Shading: H′ anomaly (m)
    - Contours: H′ anomaly (solid=positive, dashed=negative)
    - Solid light gray contours: mean field (H) for reference
    - Wind vectors: mean winds
    - Significant areas (p<0.05): stippling
    """
    # Convert to numpy arrays
    lon = np.array(lon)
    lat = np.array(lat)
    Hp = np.array(Hp)
    U = np.array(U)
    V = np.array(V)
    sig = np.array(sig)

    # --- Figure & projection ---
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(7.4, 5.0), dpi=200, subplot_kw={'projection': proj})
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)

    # --- Base map ---
    ax.coastlines(resolution="50m", linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

    # --- Shaded H′ anomaly ---
    norm = TwoSlopeNorm(vcenter=0)
    pcm = ax.pcolormesh(lon, lat, Hp, cmap="RdBu_r", norm=norm,
                        shading="auto", transform=proj)

    # --- Contours (solid positive, dashed negative) ---
    pos = np.arange(0, np.nanmax(Hp), 5)
    neg = np.arange(np.nanmin(Hp), 0, 5)
    if len(pos) > 0:
        ax.contour(lon, lat, Hp, levels=pos, colors="k",
                   linewidths=0.5, linestyles="solid", transform=proj)
    if len(neg) > 0:
        ax.contour(lon, lat, Hp, levels=neg, colors="k",
                   linewidths=0.5, linestyles="dashed", transform=proj)

    # --- Wind vectors ---
    step = 4
    ax.quiver(lon[::step, ::step], lat[::step, ::step],
              U[::step, ::step], V[::step, ::step],
              scale=700, width=0.002, color="black", transform=proj)

    # --- Stippling ---
    y, x = np.where(sig)
    thin = 8  # Thinning factor: only plot 1 of every N significant points
    y = y[::thin]
    x = x[::thin]
    ax.scatter(lon[y, x], lat[y, x], s=2, c="k", alpha=0.25, transform=proj)


    # --- MCMA box ---
    rect = mpatches.Rectangle((MCMA_BOX[0], MCMA_BOX[1]),
                              MCMA_BOX[2], MCMA_BOX[3],
                              fill=False, edgecolor="k", linewidth=1, transform=proj)
    ax.add_patch(rect)

    # --- CDMX star ---
    ax.plot(LON_CDMX, LAT_CDMX, marker="*", color="gold",
            markersize=9, markeredgecolor="k", transform=proj)

    # --- Grid & labels ---
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray",
                      alpha=0.5, linestyle="--")
    gl.top_labels = gl.right_labels = False

    # --- Colorbar ---
    cb = fig.colorbar(pcm, ax=ax, shrink=0.8, pad=0.04)
    cb.set_label("Geopotential Height Anomaly (H′, m)", fontsize=9)
    cb.ax.tick_params(labelsize=8)

    # --- Title ---
    ax.set_title(f"{pollutant}: {level}-hPa H′ (anomaly, contours & wind vectors)",
                 fontsize=12, weight="bold")

    # --- Saving ---
    outname = f"map_{pollutant}_{level}hPa.png"
    plt.savefig(outname, dpi=300)
    plt.close(fig)
    print(f"Saved: {outname}")

## 5. H = 500 hPa composites

In [9]:
# Compute H′ anomalies relative to daily climatology
# Using groupby to align daily climatology (dayofyear) with the time dimension
Hprime500 = H500.groupby("time.dayofyear") - clim500

for pol in pollutants:
    print(f" → {pol}")
    dates = events[pol]
    Hm = composite(H500, dates)
    Hp = composite(Hprime500, dates)
    Um = composite(U500, dates)
    Vm = composite(V500, dates)
    sig = ttest_mask(Hprime500, dates)
    plot_composite(lon2d, lat2d, Hm, Hp, Um, Vm, sig, pol, 500)

 → PM2.5
Saved: map_PM2.5_500hPa.png
 → PM10
Saved: map_PM10_500hPa.png
 → O3
Saved: map_O3_500hPa.png
 → NO2
Saved: map_NO2_500hPa.png
 → SO2
Saved: map_SO2_500hPa.png


In [ ]:
# --- Multipanel layout for 500 hPa composites ---
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5),
                         subplot_kw={'projection': ccrs.PlateCarree()},
                         dpi=250)
axes = axes.flatten()

for i, pol in enumerate(pollutants):
    if i >= len(pollutants):
        axes[i].axis("off")
        continue

    ax = axes[i]
    print(f"Rendering {pol} (500 hPa)...")

    # Get event composites
    dates = events[pol]
    Hm = composite(H500, dates)
    Hp = composite(Hprime500, dates)
    Um = composite(U500, dates)
    Vm = composite(V500, dates)
    sig = ttest_mask(Hprime500, dates)

    # Convert to numpy
    Hp = np.array(Hp)
    U = np.array(Um)
    V = np.array(Vm)
    sig = np.array(sig)

    # --- Base map ---
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

    # --- H′ shading ---
    norm = TwoSlopeNorm(vcenter=0)
    pcm = ax.pcolormesh(lon2d, lat2d, Hp, cmap="RdBu_r",
                        norm=norm, shading="auto", transform=ccrs.PlateCarree())

    # --- Contours (solid pos / dashed neg) ---
    pos = np.arange(0, np.nanmax(Hp), 5)
    neg = np.arange(np.nanmin(Hp), 0, 5)
    if len(pos) > 0:
        ax.contour(lon2d, lat2d, Hp, levels=pos, colors="k",
                   linewidths=0.5, linestyles="solid", transform=ccrs.PlateCarree())
    if len(neg) > 0:
        ax.contour(lon2d, lat2d, Hp, levels=neg, colors="k",
                   linewidths=0.5, linestyles="dashed", transform=ccrs.PlateCarree())

    # --- Wind vectors ---
    step = 4
    ax.quiver(lon2d[::step, ::step], lat2d[::step, ::step],
              U[::step, ::step], V[::step, ::step],
              scale=700, width=0.002, color="black", transform=ccrs.PlateCarree())

    # --- Stippling (clean density) ---
    y, x = np.where(sig)
    thin = 8
    y = y[::thin]
    x = x[::thin]
    ax.scatter(lon2d[y, x], lat2d[y, x], s=2, c="k", alpha=0.25, transform=ccrs.PlateCarree())

    # --- CDMX box & star ---
    rect = mpatches.Rectangle((MCMA_BOX[0], MCMA_BOX[1]), MCMA_BOX[2], MCMA_BOX[3],
                              fill=False, edgecolor="k", linewidth=1, transform=ccrs.PlateCarree())
    ax.add_patch(rect)
    ax.plot(LON_CDMX, LAT_CDMX, marker="*", color="gold",
            markersize=8, markeredgecolor="k", transform=ccrs.PlateCarree())

    # --- Title per panel ---
    ax.set_title(pol, fontsize=10, weight="bold")

# Remove empty panel if only 5 pollutants
if len(pollutants) < 6:
    axes[-1].axis("off")

# --- Shared colorbar ---
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
cb = fig.colorbar(pcm, cax=cbar_ax, label="Geopotential Height Anomaly (H′, m)")

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig("layout_500hPa_final.png", dpi=300)
plt.show()

Rendering PM2.5 (500 hPa)...
Rendering PM10 (500 hPa)...
Rendering O3 (500 hPa)...
Rendering NO2 (500 hPa)...
Rendering SO2 (500 hPa)...


C:\Users\DELL\AppData\Local\Temp\ipykernel_17100\4244931976.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
C:\Users\DELL\AppData\Local\Temp\ipykernel_17100\4244931976.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Understanding the climate pattern better

##### Diagnosis with aggregated data & analyzing wind direction and speed

In [21]:
# --- General helpers for the NARR-like DataArrays ---
def mean_Hprime_over_box(Hprime, dates, box):
    # Extracting coordinate names
    lat_name = [n for n in Hprime.coords if "lat" in n.lower()][0]
    lon_name = [n for n in Hprime.coords if "lon" in n.lower()][0]
    lat = Hprime[lat_name]
    lon = Hprime[lon_name]

    # Unpacking box
    lon0, lat0, w, h = box
    mask = (lon >= lon0) & (lon <= lon0 + w) & (lat >= lat0) & (lat <= lat0 + h)

    # Selecting time and spatial average
    evt = Hprime.sel(time=Hprime.time.isin(pd.to_datetime(dates)))
    return float(evt.where(mask).mean().values)


def mean_wind_over_box(U, V, dates, box):
    # Getting coordinate names automatically
    lat_name = [n for n in U.coords if "lat" in n.lower()][0]
    lon_name = [n for n in U.coords if "lon" in n.lower()][0]
    lat = U[lat_name]
    lon = U[lon_name]

    lon0, lat0, w, h = box
    mask = (lon >= lon0) & (lon <= lon0 + w) & (lat >= lat0) & (lat <= lat0 + h)

    evtU = U.sel(time=U.time.isin(pd.to_datetime(dates)))
    evtV = V.sel(time=V.time.isin(pd.to_datetime(dates)))

    Um = float(evtU.where(mask).mean().values)
    Vm = float(evtV.where(mask).mean().values)
    speed = (Um**2 + Vm**2) ** 0.5
    direction = (np.degrees(np.arctan2(Um, Vm)) % 360)
    return speed, direction


# --- Running both diagnostics ---
summary = {p: mean_Hprime_over_box(Hprime500, events[p], MCMA_BOX)
           for p in pollutants}
print("\nMean H′ (500 hPa) over MCMA:")
print(pd.Series(summary, name="H′ (m)").round(2))

print("\nMean wind speed and direction (500 hPa) over MCMA:")
for p in pollutants:
    s, d = mean_wind_over_box(U500, V500, events[p], MCMA_BOX)
    print(f"{p:5s}: {s:.2f} m/s, dir = {d:.0f}°")



Mean H′ (500 hPa) over MCMA:
PM2.5   4.17
PM10    6.10
O3      5.58
NO2     4.51
SO2     4.44
Name: H′ (m), dtype: float64

Mean wind speed and direction (500 hPa) over MCMA:
PM2.5: 1.70 m/s, dir = 61°
PM10 : 0.91 m/s, dir = 43°
O3   : 0.75 m/s, dir = 56°
NO2  : 1.35 m/s, dir = 72°
SO2  : 0.78 m/s, dir = 71°


##### Monthly distribution of event days (for context)

In [15]:
import calendar

# Count how many event days fall in each month per pollutant
monthly_counts = {
    p: pd.to_datetime(pd.Series(events[p])).dt.month.value_counts().sort_index()
    for p in pollutants
}

fig, ax = plt.subplots(figsize=(9,5))
for p in pollutants:
    months = monthly_counts[p].index
    values = monthly_counts[p].values
    ax.plot(months, values, marker="o", label=p)

ax.set_xticks(range(1,13))
ax.set_xticklabels(calendar.month_abbr[1:], rotation=0)
ax.set_xlabel("Month")
ax.set_ylabel("Number of high-pollution days")
ax.set_title("Monthly distribution of peak events (p90 per month, 2012–2024)")
ax.legend(title="Pollutant", frameon=False)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("monthly_distribution_events.png", dpi=300, bbox_inches="tight")
plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_17100\1578329090.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
